# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is documented by a Croissant schema, accessible at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata details
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")


## 2. Data Overview
Let's review record sets defined in the Croissant schema. Each `RecordSet` can be referenced by its `@id`.

In [ ]:
# List all record sets and their fields by `@id`
record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No record sets are defined explicitly in the metadata. Checking for implicit record sets from distributions...")
    # Explore distributions to check for available files
    for dist in dataset.metadata.distributions:
        print(f"Distribution @id: {dist['@id']} -- URL: {dist.get('contentUrl')}")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        # Fields linked to this record set by their @id
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        print("Fields (@id):")
        for field in fields:
            print(f"  - {field if isinstance(field, str) else field.get('@id', '')}")

## 3. Data Extraction
We can extract data from a record set by referencing its `@id`. If no explicit `recordSet` is present, we use the dataset's primary data table instead.

*For this dataset, the distribution data file is the data table to be explored. Let's get its `@id` and load it as records.*

In [ ]:
# Find main data record set or use the main distribution's @id as the record set
if dataset.metadata.record_sets:
    # Use the first record set (or adjust for your dataset as needed)
    main_record_set_id = dataset.metadata.record_sets[0]['@id']
else:
    # No explicit record set: use first distribution @id as record set for loading records
    main_record_set_id = dataset.metadata.distributions[0]['@id']
    print(f"Using main distribution as record set: {main_record_set_id}")

# For future reference, list all record set/distribution ids:
all_record_set_ids = []
if dataset.metadata.record_sets:
    all_record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
else:
    all_record_set_ids = [dist['@id'] for dist in dataset.metadata.distributions]

# Load data as DataFrame for each record set/ distribution id
dataframes = {}
for rsid in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for record_set/distribution: {rsid}  shape: {df.shape}")
    except Exception as e:
        print(f"Error loading record_set {rsid}: {e}")

df = dataframes[main_record_set_id]
print("\nAvailable columns (field @id or schema keys):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main data table, select a numeric field (by column name as `@id`), filter records, and normalize values.

Suppose a common numeric field for mass spectrometry proteins is `MW [kDa]` (molecular weight, in kiloDaltons) or `Peptides` (peptide count). The exact column names may differ--refer to those printed above. For this demonstration, update the variable below to a field you see in your columns.

In [ ]:
# Edit these according to the printed columns, using exact column (field/@id) names
numeric_field_id = None
# Attempt to guess field (edit below as needed)
for col in df.columns:
    if 'MW' in col or 'Peptide' in col or 'Abundance' in col:
        numeric_field_id = col
        break
if not numeric_field_id:
    numeric_field_id = df.columns[df.dtypes == float].tolist()[0] if (df.dtypes == float).any() else df.columns[0]
print(f"Analyzing numeric field: {numeric_field_id}")

# Filtering (e.g., molecular weight > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered rows where {numeric_field_id} > {threshold} (count: {filtered_df.shape[0]}):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field_id + '_normalized'] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"\nNormalized field {numeric_field_id} (first few rows):")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Optional: Group by a categorical field (edit group_field_id to match your data)
group_field_id = None
for col in df.columns:
    if 'Group' in col or 'Sample' in col or 'Category' in col or 'Cell' in col:
        group_field_id = col
        break
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped (mean) {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected numeric field and, if available, show boxplots for groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True, bins=30)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by group (if available)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We loaded and inspected a FAIR^2 mass spectrometry dataset using Croissant metadata.
- Data was filtered and normalized for a numeric field (e.g., molecular weight or abundance).
- Simple visualizations illustrated distribution and potential grouping.

This notebook provides an extensible template for working with other Croissant-based datasets, ensuring all record sets, fields, and data are referenced by their `@id` as per FAIR best practices.